# CMIP6 core fix-plan exampleThis notebook uses a tiny synthetic CMIP6 dataset and a YAML fix plan to show the public Woodpecker plan API: inspect plan content, check a dataset with a plan, dry-run the plan, apply it in memory, and re-check the result.The plan file is shared with integration tests (`tests/integration/plans/cmip6_core_plan.yaml`).

In [ ]:
import json
from IPython.display import Markdown, display

import numpy as np
import yaml

import woodpecker
from woodpecker.testing import integration_plan_path, make_cmip6


In [ ]:
def show_json(value):
    display(Markdown("```json\n" + json.dumps(value, indent=2, sort_keys=False) + "\n```"))

def show_yaml(value):
    display(Markdown("```yaml\n" + yaml.safe_dump(value, sort_keys=False) + "```"))

def show_dataset_summary(dataset, title="Dataset summary"):
    summary = {
        "dims": {k: int(v) for k, v in dataset.sizes.items()},
        "coords": list(dataset.coords),
        "data_vars": list(dataset.data_vars),
        "attrs": dict(dataset.attrs),
    }
    display(Markdown(f"**{title}**"))
    show_yaml(summary)


Create a deterministic CMIP6-like dataset where near-surface air temperature is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

show_dataset_summary(dataset, title="Input xarray dataset")
dataset[["tas"]].isel(time=slice(0, 2), lat=slice(0, 2), lon=slice(0, 2))


Load the shared YAML fix-plan document. The plan matches CMIP6-like datasets and selects the built-in core units fix.Show the parsed YAML plan content directly in notebook output.

In [ ]:
plan_path = integration_plan_path("cmip6_core_plan.yaml")
plan_content = yaml.safe_load(plan_path.read_text(encoding="utf-8"))

display(Markdown(f"**Plan file:** `{plan_path.name}`"))
show_yaml(plan_content)


In [ ]:
findings = woodpecker.check_plan(plan_path, inputs=dataset)

show_json({"has_findings": findings.has_findings, "fix_ids": findings.fix_ids})


A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.fix_plan(plan_path, inputs=dataset, write=False)

show_json(
    {
        "stats": dry_run.stats,
        "units": dataset["tas"].attrs["units"],
        "values_unchanged": bool(np.allclose(dataset["tas"].values, original_values)),
    }
)


Apply the same plan in memory with `write=True`.

In [ ]:
write = woodpecker.fix_plan(plan_path, inputs=dataset, write=True)

show_json(
    {
        "stats": write.stats,
        "units": dataset["tas"].attrs["units"],
        "converted_to_kelvin": bool(np.allclose(dataset["tas"].values, original_values + 273.15)),
    }
)


In [ ]:
recheck = woodpecker.check_plan(plan_path, inputs=dataset)

show_json({"has_findings": recheck.has_findings})
